In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [3]:
load_dotenv()

True

In [4]:
model = ChatOpenAI(model = 'gpt-4o-mini')

In [5]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed Feedback for the essay')
    score: int = Field(description='Score out of 10', ge=10, le=10)


In [6]:
structured_model = model.with_structured_output(EvaluationSchema)

In [7]:
essay = """Artificial Intelligence (AI) is transforming the global economy, and India is emerging as a significant player in this technological revolution. With its strong IT industry, vast talent pool, and growing digital infrastructure, India is leveraging AI to drive innovation, economic growth, and social development.

One of India’s key strengths in AI is its skilled workforce. The country produces a large number of engineers, data scientists, and software professionals every year. Indian institutes such as the IITs, IISc, and other leading universities actively conduct research in AI, machine learning, and data science, contributing to global advancements in the field.

The Indian government plays an important role in promoting AI adoption. Initiatives like Digital India, National Strategy for Artificial Intelligence, and IndiaAI Mission focus on using AI for inclusive growth. These programs aim to apply AI in critical sectors such as healthcare, agriculture, education, smart cities, and governance, ensuring that technological progress benefits society at large.

India’s IT and startup ecosystem has also accelerated AI development. Indian technology companies and startups are building AI-powered solutions in areas such as fintech, e-commerce, cybersecurity, language processing, and automation. Many global organizations rely on Indian firms for AI research, development, and implementation, strengthening India’s position in the global AI value chain.

AI is increasingly being used to address India’s social and economic challenges. In agriculture, AI helps farmers with crop prediction, pest detection, and efficient resource usage. In healthcare, AI supports early disease detection, medical imaging, and telemedicine, improving access to quality healthcare in remote areas. AI-based governance tools enhance public service delivery, transparency, and decision-making.

India also emphasizes ethical and responsible AI. Policymakers and researchers are working to ensure fairness, transparency, and data privacy in AI systems. Given India’s diversity, developing unbiased and inclusive AI solutions is a critical priority.

In conclusion, India plays a vital role in the global AI landscape by combining technological expertise with a focus on inclusive development. Through government initiatives, academic research, industry innovation, and ethical practices, India is not only advancing AI technology but also shaping its use for social good. As AI continues to evolve, India is well-positioned to be a global leader in responsible and impactful AI adoption."""

In [9]:
prompt = f'Evaluate the quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt).score

10

In [16]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    depth_of_analysis: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [22]:
def evaluate_language(state: UPSCState):
    prompt =  f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_analysis(state: UPSCState):
    prompt =  f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [23]:
def evaluate_thought(state: UPSCState):
    prompt =  f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [26]:
def final_evaluation(state: UPSCState):

    #summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback- {state['language_feedback']} \n clarity of thought- {state['clarity_feedback']} \n depth of analysis- {state['depth_of_analysis']}'
    final_feedback = model.invoke(prompt).content
    #avg calculate
    sum(state['individual_scores'])/ len(state['individual_scores'])

    return {'overall_feedback': final_feedback, 'avg_score': avg_score} 

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

#edges 
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)
